In [ ]:
# INSTALL & IMPORT LIBRARY

import os
import shutil
import zipfile
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# KONFIGURASI GLOBAL

FOLDER_PREPROCESSING = '/content/drive/MyDrive/SolarPanel_Preprocessing'
PATH_ZIP_TILE        = os.path.join(FOLDER_PREPROCESSING, 'AllTiles_backup.zip')
PATH_CSV_MENTAH      = os.path.join(FOLDER_PREPROCESSING, 'parameter_mentah_per_tile.csv')

FOLDER_TILE_LOKAL = '/content/AllTiles_local'
FOLDER_DATASET     = '/content/SolarPanel_Dataset'
FOLDER_OUTPUT_DRIVE = '/content/drive/MyDrive/SolarPanel_Label'

KELAS_LC_POTENTIAL = [2, 6, 7]
SLOPE_MAKS_DERAJAT = 5.0
GHI_MIN_KWH = 4.5


RANDOM_SEED = 42

os.makedirs(FOLDER_TILE_LOKAL, exist_ok=True)
os.makedirs(FOLDER_OUTPUT_DRIVE, exist_ok=True)

In [ ]:
# EKSTRAK TILE DARI ZIP & MUAT METADATA CSV

print('Mengekstrak tile dari zip hasil Tahap 1 Preprocessing...')
with zipfile.ZipFile(PATH_ZIP_TILE, 'r') as zip_ref:
    zip_ref.extractall(FOLDER_TILE_LOKAL)
print(f'✓ Tile berhasil diekstrak ke: {FOLDER_TILE_LOKAL}')

df_mentah = pd.read_csv(PATH_CSV_MENTAH)
print(f'✓ Metadata dimuat: {len(df_mentah)} tile')
print(f'\nKolom yang tersedia:')
print(df_mentah.columns.tolist())

Mengekstrak tile dari zip hasil Tahap 1 Preprocessing...
✓ Tile berhasil diekstrak ke: /content/AllTiles_local
✓ Metadata dimuat: 4705 tile

Kolom yang tersedia:
['filename', 'kota', 'baris', 'kolom', 'kelas_lc_kode', 'kelas_lc_nama', 'slope_rata2', 'ghi_rata2', 'kosong_rgb', 'kosong_dw', 'kosong_slope', 'kosong_ghi']


In [ ]:
# TERAPKAN RULE-BASED LABELING OTOMATIS (v3 - 3 kelas)

def terapkan_label_otomatis(row):
    """
    Menerapkan rule-based labeling berdasarkan 3 parameter:
    tutupan lahan, slope, dan GHI.

    Logika v3:
    - lc adalah SATU-SATUNYA parameter yang punya representasi visual
      langsung di citra RGB (input CNN). Slope & GHI tidak terlihat
      secara visual, jadi kalau lc gagal tapi slope/ghi lolos (atau
      sebaliknya), itu bukan konflik visual - lc yang gagal sudah
      cukup jadi sinyal Not_Potential yang konsisten dengan citra.
    - Konflik HANYA terjadi saat lc lolos (visual bilang "berpotensi")
      tapi slope/ghi menggagalkannya (label bilang "tidak") - kasus
      ini dilabel 'Ambigu' dan nanti dikeluarkan dari data training CNN.
    """
    syarat_lc    = row['kelas_lc_kode'] in KELAS_LC_POTENTIAL
    syarat_slope = row['slope_rata2']   <= SLOPE_MAKS_DERAJAT
    syarat_ghi   = row['ghi_rata2']     >= GHI_MIN_KWH

    if not syarat_lc:
        label = 'Not_Potential'
    elif syarat_slope and syarat_ghi:
        label = 'Potential'
    else:
        label = 'Ambigu'

    return pd.Series([label, syarat_lc, syarat_slope, syarat_ghi])


df_mentah[['label_otomatis', 'syarat_lc', 'syarat_slope', 'syarat_ghi']] = \
    df_mentah.apply(terapkan_label_otomatis, axis=1)

print(f'{"="*60}')
print('HASIL PELABELAN OTOMATIS (v3 - 3 kelas)')
print(f'{"="*60}')
print(df_mentah['label_otomatis'].value_counts())
print(f'\nDistribusi per kota:')
print(df_mentah.groupby(['kota', 'label_otomatis']).size().unstack(fill_value=0))


HASIL PELABELAN OTOMATIS (v3 - 3 kelas)
label_otomatis
Not_Potential    2782
Potential        1723
Ambigu            200
Name: count, dtype: int64

Distribusi per kota:
label_otomatis  Ambigu  Not_Potential  Potential
kota                                            
BandaAceh            2             82         33
Bandung             84            331         98
Denpasar             6            201         97
Jakarta             76            688        662
Jayapura             5             24          1
Kediri               5            308        219
Makassar             1            213         71
Pontianak            0            159         33
Surabaya            11            380        233
Yogyakarta          10            396        276


In [ ]:
# KELUARKAN KELAS 'Ambigu' DARI DATA TRAINING CNN
# Ambigu = lc memenuhi tapi slope/ghi tidak (konflik visual vs label).
# Disimpan terpisah untuk dokumentasi/lampiran skripsi, TIDAK dipakai
# untuk training/val/test CNN.

df_ambigu  = df_mentah[df_mentah['label_otomatis'] == 'Ambigu'].reset_index(drop=True)
df_mentah  = df_mentah[df_mentah['label_otomatis'] != 'Ambigu'].reset_index(drop=True)

print(f'Tile Ambigu dikeluarkan: {len(df_ambigu)} ({len(df_ambigu)/(len(df_mentah)+len(df_ambigu))*100:.2f}%)')
print(f'Tile bersih untuk training CNN: {len(df_mentah)}')
print(df_mentah['label_otomatis'].value_counts())

path_csv_ambigu = os.path.join(FOLDER_OUTPUT_DRIVE, 'metadata_ambigu_dikeluarkan.csv')
df_ambigu.to_csv(path_csv_ambigu, index=False)
print(f"\u2713 Metadata tile Ambigu disimpan (dokumentasi) ke: {path_csv_ambigu}")


Tile Ambigu dikeluarkan: 200 (4.25%)
Tile bersih untuk training CNN: 4505
label_otomatis
Not_Potential    2782
Potential        1723
Name: count, dtype: int64
✓ Metadata tile Ambigu disimpan (dokumentasi) ke: /content/drive/MyDrive/SolarPanel_Label/metadata_ambigu_dikeluarkan.csv


In [ ]:
# FUNGSI & PROSES AUGMENTASI
from PIL import Image
import random

def augmentasi_flip_horizontal(citra_array):
    return np.fliplr(citra_array)

def augmentasi_flip_vertikal(citra_array):
    return np.flipud(citra_array)

def augmentasi_rotasi(citra_array, sudut_derajat):
    k = int(sudut_derajat / 90)
    return np.rot90(citra_array, k=k)

def augmentasi_kecerahan(citra_array, faktor=1.2):
    hasil = citra_array.astype(np.float32) * faktor
    return np.clip(hasil, 0, 255).astype(np.uint8)

DAFTAR_KOMBINASI_AUGMENTASI = [
    lambda img: augmentasi_flip_horizontal(img),
    lambda img: augmentasi_flip_vertikal(img),
    lambda img: augmentasi_rotasi(img, 90),
    lambda img: augmentasi_rotasi(img, 180),
    lambda img: augmentasi_rotasi(img, 270),
    lambda img: augmentasi_kecerahan(img, 1.2),
    lambda img: augmentasi_kecerahan(img, 0.85),
    lambda img: augmentasi_flip_horizontal(augmentasi_rotasi(img, 90)),
]

def seimbangkan_dengan_augmentasi(df_train, folder_sumber):
    jml_pot = (df_train['label_otomatis'] == 'Potential').sum()
    jml_not = (df_train['label_otomatis'] == 'Not_Potential').sum()

    df_train = df_train.copy()
    df_train['is_augmentasi'] = False

    if jml_pot == jml_not:
        print('Train sudah seimbang, tidak perlu augmentasi.')
        return df_train

    kelas_minor  = 'Potential' if jml_pot < jml_not else 'Not_Potential'
    jumlah_kurang = abs(jml_pot - jml_not)

    df_minor = df_train[df_train['label_otomatis'] == kelas_minor].reset_index(drop=True)
    print(f'Kelas minor di train: {kelas_minor} | perlu +{jumlah_kurang} tile sintetis')

    baris_baru = []
    idx_sumber = 0
    idx_kombinasi = 0

    while len(baris_baru) < jumlah_kurang:
        row_asli   = df_minor.iloc[idx_sumber % len(df_minor)]
        path_asli  = os.path.join(folder_sumber, row_asli['filename'])
        img        = np.array(Image.open(path_asli))

        fungsi_aug = DAFTAR_KOMBINASI_AUGMENTASI[idx_kombinasi % len(DAFTAR_KOMBINASI_AUGMENTASI)]
        img_aug    = fungsi_aug(img)

        nama_baru = f"{row_asli['filename'].replace('.png','')}_aug{idx_kombinasi % len(DAFTAR_KOMBINASI_AUGMENTASI)}_{idx_sumber}.png"
        Image.fromarray(img_aug).save(os.path.join(folder_sumber, nama_baru))

        row_baru = row_asli.copy()
        row_baru['filename'] = nama_baru
        row_baru['is_augmentasi'] = True
        baris_baru.append(row_baru)

        idx_sumber += 1
        idx_kombinasi += 1   # <-- FIX: increment SETIAP iterasi (round-robin), bukan menunggu 1 putaran source habis

    df_train_final = pd.concat([df_train, pd.DataFrame(baris_baru)], ignore_index=True)
    print(f'✓ Train setelah augmentasi: {df_train_final["label_otomatis"].value_counts().to_dict()}')
    return df_train_final

In [ ]:
# Split blok spasial via greedy bin-packing, 70/20/10
# TANPA augmentasi dulu (raw split)
import numpy as np

FOLDER_DRIVE_DATASET = os.path.join(FOLDER_OUTPUT_DRIVE, 'Dataset_Split_Final_v2_noaug')

subfolders = [
    'train/Potential', 'train/Not_Potential',
    'val/Potential', 'val/Not_Potential',
    'test/Potential', 'test/Not_Potential'
]
for sf in subfolders:
    os.makedirs(os.path.join(FOLDER_DRIVE_DATASET, sf), exist_ok=True)

BLOCK_SIZE = 3
df_mentah['block_id'] = (
    df_mentah['kota'] + '_' +
    (df_mentah['baris'] // BLOCK_SIZE).astype(str) + '_' +
    (df_mentah['kolom'] // BLOCK_SIZE).astype(str)
)

block_info = (
    df_mentah.groupby('block_id')['label_otomatis']
    .value_counts().unstack(fill_value=0)
)
for kelas in ['Potential', 'Not_Potential']:
    if kelas not in block_info.columns:
        block_info[kelas] = 0
block_info['total'] = block_info['Potential'] + block_info['Not_Potential']
block_info = block_info.sample(frac=1, random_state=RANDOM_SEED)

total_tile = len(df_mentah)
target = {'train': 0.70*total_tile, 'val': 0.20*total_tile, 'test': 0.10*total_tile}
current = {'train': 0, 'val': 0, 'test': 0}
block_to_split = {}

for block_id, row in block_info.iterrows():
    defisit = {s: target[s] - current[s] for s in current}
    split_pilihan = max(defisit, key=defisit.get)
    block_to_split[block_id] = split_pilihan
    current[split_pilihan] += row['total']

df_mentah['split_baru'] = df_mentah['block_id'].map(block_to_split)

print('Rasio aktual:')
for s in ['train', 'val', 'test']:
    jumlah = (df_mentah['split_baru'] == s).sum()
    print(f'  {s}: {jumlah} tile ({jumlah/total_tile*100:.2f}%)')

print('\nKeseimbangan kelas per split:')
print(df_mentah.groupby('split_baru')['label_otomatis'].value_counts().unstack(fill_value=0))

cek = df_mentah.groupby('block_id')['split_baru'].nunique()
print('\nJumlah blok bocor lintas split (harus 0):', (cek > 1).sum())

df_train = df_mentah[df_mentah['split_baru'] == 'train'].sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
df_val   = df_mentah[df_mentah['split_baru'] == 'val'].sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
df_test  = df_mentah[df_mentah['split_baru'] == 'test'].sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)


df_val['is_augmentasi']   = False
df_test['is_augmentasi']  = False

df_train = seimbangkan_dengan_augmentasi(df_train, FOLDER_TILE_LOKAL)

df_train['split'] = 'train'
df_val['split']   = 'val'
df_test['split']  = 'test'

df_final_lengkap = pd.concat([df_train, df_val, df_test]).reset_index(drop=True)

def salin_file_ke_drive(df, nama_split):
    berhasil = 0
    for _, row in df.iterrows():
        src = os.path.join(FOLDER_TILE_LOKAL, row['filename'])
        label = row['label_otomatis']
        dst = os.path.join(FOLDER_DRIVE_DATASET, nama_split, label, row['filename'])
        if os.path.exists(src):
            shutil.copy(src, dst)
            berhasil += 1
    print(f"✓ Berhasil menyalin {berhasil} file ke folder '{nama_split}' di Drive.")

print('\nMenyalin file ke Drive...')
salin_file_ke_drive(df_train, 'train')
salin_file_ke_drive(df_val, 'val')
salin_file_ke_drive(df_test, 'test')

path_csv_output = os.path.join(FOLDER_OUTPUT_DRIVE, 'metadata_labeling_v2_noaug.csv')
df_final_lengkap.to_csv(path_csv_output, index=False)
print(f'✓ Metadata CSV disimpan ke: {path_csv_output}')

Rasio aktual:
  train: 3150 tile (69.92%)
  val: 906 tile (20.11%)
  test: 449 tile (9.97%)

Keseimbangan kelas per split:
label_otomatis  Not_Potential  Potential
split_baru                              
test                      303        146
train                    1910       1240
val                       569        337

Jumlah blok bocor lintas split (harus 0): 0
Kelas minor di train: Potential | perlu +670 tile sintetis
✓ Train setelah augmentasi: {'Potential': 1910, 'Not_Potential': 1910}

Menyalin file ke Drive...
✓ Berhasil menyalin 3820 file ke folder 'train' di Drive.
✓ Berhasil menyalin 906 file ke folder 'val' di Drive.
✓ Berhasil menyalin 449 file ke folder 'test' di Drive.
✓ Metadata CSV disimpan ke: /content/drive/MyDrive/SolarPanel_Label/metadata_labeling_v2_noaug.csv
